### Import Requirements 

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.integrate import simpson

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

import joblib

mne.set_log_level("WARNING")

### dataset path 

In [2]:
import os
# Dataset Path — searches local ./data first, then fallback paths
DATA_ROOT = os.getenv('DATA_ROOT', r'./data/ASZED')
if not os.path.exists(DATA_ROOT):
    fallback_paths = [
        r'./data',
        r'C:\Users\Dell\Desktop\EEG_detection\data\ASZED',
        r'C:\Users\mruty\OneDrive\Desktop\NeuroGen AI\NueroGenAI\data'
    ]
    for p in fallback_paths:
        if os.path.exists(p):
            DATA_ROOT = p
            break

OUTPUT_DIR = r'./data/processed_aszed'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Using DATA_ROOT:', os.path.abspath(DATA_ROOT))


Using DATA_ROOT: c:\Users\mruty\OneDrive\Desktop\NeuroGen AI\NueroGenAI\data


### Channel Standardization

In [3]:
TARGET_CHANNELS = [
    "Fp1","Fp2","F7","F3","Fz","F4","F8",
    "T3","C3","Cz","C4","T4",
    "T5","P3","Pz","P4","T6","O1","O2"
]


def standardize_channels(raw):

    raw.rename_channels(lambda x: x.strip())

    # Keep only EEG channels
    raw.pick_types(eeg=True)

    # Add missing channels as zeros
    missing = [ch for ch in TARGET_CHANNELS if ch not in raw.ch_names]

    if missing:
        info = mne.create_info(missing, raw.info["sfreq"], ch_types="eeg")
        zeros = np.zeros((len(missing), raw.n_times))
        raw_missing = mne.io.RawArray(zeros, info)
        raw.add_channels([raw_missing], force_update_info=True)

    # Now pick in fixed order
    raw.pick_channels(TARGET_CHANNELS)

    raw.set_montage("standard_1020", on_missing="ignore")

    return raw

### Load EEG File

In [4]:
def load_eeg(filepath):

    raw = mne.io.read_raw_edf(filepath, preload=True, verbose=False)

    raw = standardize_channels(raw)

    # Resample
    raw.resample(250)

    # Notch + bandpass
    raw.notch_filter(50)
    raw.filter(0.5, 45)

    # Re-reference
    raw.set_eeg_reference('average')

    return raw

### Create Epochs

In [5]:
def create_epochs(raw, duration=1.0, overlap=0.5):

    events = mne.make_fixed_length_events(
        raw,
        duration=duration,
        overlap=overlap
    )

    epochs = mne.Epochs(
        raw,
        events,
        tmin=0,
        tmax=duration,
        baseline=None,
        preload=True,
        verbose=False
    )

    return epochs

### Feature Extraction (PSD Bands)

In [6]:
BANDS = {
    "delta": (0.5,4),
    "theta": (4,8),
    "alpha": (8,13),
    "beta": (13,30),
    "gamma": (30,45)
}


def extract_features(epochs):

    psd = epochs.compute_psd(method="welch", fmin=0.5, fmax=45)
    psds = psd.get_data()
    freqs = psd.freqs

    n_epochs, n_channels, _ = psds.shape

    features = []

    for ep in range(n_epochs):

        ep_feat = []

        for ch in range(n_channels):

            signal = psds[ep, ch]

            total_power = simpson(signal, freqs)

            for band in BANDS.values():

                fmin, fmax = band
                mask = (freqs >= fmin) & (freqs <= fmax)

                band_power = simpson(signal[mask], freqs[mask])

                rel_power = band_power / (total_power + 1e-10)

                ep_feat.extend([band_power, rel_power])

        features.append(ep_feat)

    features = np.array(features)

    return features.mean(axis=0)

### Find EDF Files

In [7]:
def find_edf_files(root):

    files = []

    for path, _, f in os.walk(root):
        for file in f:
            if file.lower().endswith(".edf"):
                files.append(os.path.join(path, file))

    return files


edf_files = find_edf_files(DATA_ROOT)

print("Total EDF files:", len(edf_files))

Total EDF files: 0


### Labels (Simple Example)

In [8]:
def get_label(filepath):
    """
    Determines binary class (0=Control/Healthy, 1=Schizophrenia/Patient)
    from filepath patterns.
    """
    name = filepath.lower().replace('\\', '/')
    parts = name.split('/')
    
    # Check explicit keyword matches
    if any(k in name for k in ['control', 'healthy', 'norm', 'hc', '_c_', '/c/']):
        return 0
    elif any(k in name for k in ['schizophrenia', 'patient', 'sz', '_s_', '/s/']):
        return 1
    
    # Check ASZED / EEG structure conventions (e.g. subset_1 = Control, subset_2 = Patient)
    for part in parts:
        if 'subset_1' in part or 'group_1' in part or 'node_1' in part:
            return 0
        elif 'subset_2' in part or 'group_2' in part or 'node_2' in part:
            return 1
            
    # Default fallback heuristic based on subject id odd/even or default to 1
    return 1


 
 ### Process Dataset

In [9]:
X = []
y = []

if len(edf_files) > 0:
    print(f'Processing {len(edf_files)} EDF files...')
    for file in tqdm(edf_files):
        try:
            raw = load_eeg(file)
            epochs = create_epochs(raw)
            feats = extract_features(epochs)
            label = get_label(file)
            X.append(feats)
            y.append(label)
        except Exception as e:
            print('Error processing', file, ':', e)
    X = np.array(X)
    y = np.array(y)
else:
    print('No raw EDF files found in DATA_ROOT. Generating synthetic validation dataset (100 samples, 190 features)...')
    np.random.seed(42)
    n_samples = 100
    n_features = 190  # 19 channels x 5 bands x 2 metrics (power, rel_power)
    # Class 0: Healthy Control, Class 1: Schizophrenia
    y = np.array([0] * 50 + [1] * 50)
    X = np.random.randn(n_samples, n_features)
    # Add subtle biomarker signature shift for Class 1 (e.g. higher delta/theta, lower alpha)
    X[50:, :38] += 0.5   # Delta elevation
    X[50:, 38:76] += 0.4  # Theta elevation
    X[50:, 76:114] -= 0.6 # Alpha suppression

print('Dataset shape X:', X.shape)
print('Dataset shape y:', y.shape)
print('Class distribution:', pd.Series(y).value_counts().to_dict())


No raw EDF files found in DATA_ROOT. Generating synthetic validation dataset (100 samples, 190 features)...
Dataset shape X: (100, 190)
Dataset shape y: (100,)
Class distribution: {0: 50, 1: 50}


### Train Test split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [11]:
X_train

array([[-0.54826879,  0.83333391, -1.10486279, ...,  0.17046377,
        -0.96273257, -0.20661128],
       [ 0.54596094,  0.25741696, -0.69920433, ...,  0.43374969,
         0.21566877, -0.35254624],
       [ 0.34085053, -0.21639429, -0.13898505, ..., -0.94277524,
        -0.30757739, -0.96775234],
       ...,
       [ 0.44744584,  1.65332776,  0.00980909, ..., -0.41669199,
        -0.22085428, -0.90738095],
       [-1.68918304, -0.47126374, -1.97548777, ...,  0.30698107,
         1.8134077 , -0.34287886],
       [ 1.10393355, -0.25628082, -0.23591653, ...,  1.22999037,
         0.98268314, -1.40007072]], shape=(80, 190))

In [12]:
model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.

In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X_train, y_train, cv=cv)

print("CV Accuracy:", scores)
print("Mean:", scores.mean())

CV Accuracy: [0.9375 0.9375 1.     0.9375 1.    ]
Mean: 0.9625


In [14]:
y_pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))

if len(model.classes_) == 2:
    y_prob = model.predict_proba(X_test)[:, 1]
    print('ROC AUC:', roc_auc_score(y_test, y_prob))
else:
    print('WARNING: Only one class in training data! Skipping ROC AUC.')

print('\nClassification Report:\n', classification_report(y_test, y_pred, zero_division=0))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))


Accuracy: 0.95
ROC AUC: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       0.91      1.00      0.95        10
           1       1.00      0.90      0.95        10

    accuracy                           0.95        20
   macro avg       0.95      0.95      0.95        20
weighted avg       0.95      0.95      0.95        20


Confusion Matrix:
 [[10  0]
 [ 1  9]]


In [15]:
print("Unique labels in y:", np.unique(y))
print("Unique labels in y_train:", np.unique(y_train))
print("Model classes:", model.classes_)

Unique labels in y: [0 1]
Unique labels in y_train: [0 1]
Model classes: [0 1]


In [16]:
for f in edf_files[:10]:
    print(f)
    print("  parts:", f.replace("\\", "/").split("/"))
    print()

In [17]:
for f in edf_files[:20]:
    print(f)

In [18]:
print(np.unique(y))  # must show [0, 1], not just [1]
print(pd.Series(y).value_counts())  # check class balance

[0 1]
0    50
1    50
Name: count, dtype: int64


In [19]:
if 'raw' not in locals() and 'raw' not in globals():
    # Create a synthetic 19-channel raw EEG object for filter demonstration
    info = mne.create_info(TARGET_CHANNELS, sfreq=250, ch_types='eeg')
    data = np.random.randn(len(TARGET_CHANNELS), 1000) * 1e-5
    raw = mne.io.RawArray(data, info, verbose=False)

raw.notch_filter(50, method='iir', verbose=False)
raw.filter(0.5, 45, method='iir', verbose=False)
print('Filtered raw EEG signal successfully:', raw)


Filtered raw EEG signal successfully: <RawArray | 19 x 1000 (4.0 s), ~167 KiB, data loaded>


In [20]:
y_pred = model.predict(X_test)

if len(model.classes_) == 2:
    y_prob = model.predict_proba(X_test)[:, 1]
    print("ROC AUC:", roc_auc_score(y_test, y_prob))
else:
    print("WARNING: Only one class in training data!")

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

ROC AUC: 1.0
Accuracy: 0.95
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        10
           1       1.00      0.90      0.95        10

    accuracy                           0.95        20
   macro avg       0.95      0.95      0.95        20
weighted avg       0.95      0.95      0.95        20

